In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.common.by import By

import shutil

from time import sleep

import os

import re



from selenium.webdriver.chrome.service import Service as ChromeService

from webdriver_manager.chrome import ChromeDriverManager





# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

print("Running JP FSAJP Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= 'JP FSAJP SqlReady_data_{}.xlsx'.format(str(now).replace(":",".")[:-7])

#scriptfolder = "C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\JP FSAJP"

scriptfolder=os.path.dirname(os.path.abspath(__file__))

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

#driver = webdriver.Chrome(executable_path="..\\chromedriver.exe",options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'JP FSAJP 1': 'https://www.fsa.go.jp/en/regulated/licensed/city.xls',    

        'JP FSAJP 2': 'https://www.fsa.go.jp/en/regulated/licensed/reg.xls', 

        'JP FSAJP 3': 'https://www.fsa.go.jp/en/regulated/licensed/bank_holding.xls', 

        'JP FSAJP 4': 'https://www.fsa.go.jp/en/regulated/licensed/s_banks.xls', 

        'JP FSAJP 5': 'https://www.fsa.go.jp/en/regulated/licensed/keito.xls', 

        'JP FSAJP 6': 'https://www.fsa.go.jp/en/regulated/licensed/fietb.xls', 

        'JP FSAJP 8': 'https://www.fsa.go.jp/en/regulated/licensed/fibo.xlsx', 

        'JP FSAJP 9': 'https://www.fsa.go.jp/en/regulated/licensed/fiisp.xlsx',

        'JP FSAJP 11': 'https://www.fsa.go.jp/en/regulated/licensed/ins_life.xls', 

        'JP FSAJP 12': 'https://www.fsa.go.jp/en/regulated/licensed/ins_nonlife.xls', 

        'JP FSAJP 13': 'https://www.fsa.go.jp/en/regulated/licensed/ins_holding.xls', 

        'JP FSAJP 14': 'https://www.fsa.go.jp/en/regulated/licensed/trustcompanies.xls', 

        }





Typology ={

        'JP FSAJP 1': 'City Banks and Trust Banks',    

        'JP FSAJP 2': 'Regional Banks', 

        'JP FSAJP 3': 'Bank Holding Companies', 

        'JP FSAJP 4': 'Credit Associations', 

        'JP FSAJP 5': 'Keito Financial Institutions', 

        'JP FSAJP 6': ' Financial Institutions which engage in Trust Business, etc', 

        'JP FSAJP 8': 'Financial Services Agency', 

        'JP FSAJP 9': 'Financial Instruments Intermediary Service Providers',

        'JP FSAJP 11': 'List of life insurance companies', 

        'JP FSAJP 12': 'List of non-life insurance companies (domestic companies)', 

        'JP FSAJP 13': 'Insurance holding companies', 

        'JP FSAJP 14': 'Trust Company', 

        }







sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



Headers = ["Name", "Address", "Type", "Registry Number", "Web Address", "Notes", "bvdid", "priority", "ListLabel", "Typology", "EntryType", "Name", 

           "InternalID_1", "InternalID_1_type", "InternalID_2", "InternalID_2_type", "InternalID_3", "InternalID_3_type", "CoType", "License_Type", 

           "Address_1", "Address_2", "City", "Zip", "Ctry", "Phone", "Fax", "Website", "Email", "RegulationType", "RegulationTypeCode", "RegulationDate", 

           "CancellationDate", "RegCtry", "RegCode", "ListCode", "ListLanguage", "ListValidityDate", "ListName", "ListProcessDate", "LEI Code", "BIC/SWIFT Code"]





processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict





def cleanText(text):

    '''

    This function takes a label from the li tag on the web page and removes any additional text appart from the name of

    the institution.

    Thus, text such as "(PDF * Excel)", "(Available in japanese)" are removed.

    The aim is to be able to compare the output later with the label of categories of the financial institutions to be

    retrieve from the web page's bs4 element.

    '''

    index_ = text.find("(")

    if index_ != -1:

        if text.count("(")>1:#get the number of occurence of the opening bracket

            index_ = text.find("(PDF")

            #print(text)

        text = text[:index_]

    text = text.strip()

    return text



def isStartingRowIndex(cell):

    Headers = ["Name", "Address", "Type", "Registry Number", "Web Address", "Notes", "bvdid", "priority", "ListLabel", "Typology", "EntryType", "Name", "InternalID_1", "InternalID_1_type", "InternalID_2", "InternalID_2_type", "InternalID_3", "InternalID_3_type", "CoType", "License_Type", "Address_1", "Address_2", "City", "Zip", "Ctry", "Phone", "Fax", "Website", "Email", "RegulationType", "RegulationTypeCode", "RegulationDate", "CancellationDate", "RegCtry", "RegCode", "ListCode", "ListLanguage", "ListValidityDate", "ListName", "ListProcessDate", "LEI Code", "BIC/SWIFT Code"]

    headers = [str(header).lower() for header in Headers]

    cell = str(cell).strip()

    Bool = cell.lower() in headers

    if not Bool: Bool = ("name" in cell.lower() and "unnamed" not in cell.lower())

    # if Bool: print("title detected: ", cell)

    return Bool



def getLastUpdateDate(cell):

    #elt = str(cell).lower()

    regExp = re.findall("[Aa]s of [ADFJMNOS]\w*\s*[\d]{1,2},\s*[\d]{4}|[\d]{2,4}-[\d]{2}-[\d]{2,4}", str(cell))

    if len(regExp) >0 : return regExp[0].strip("As of")

    return None



def to_lower_case(elt):

    '''

    This function puts the input to lower case

    '''

    return str(elt).lower()



def list_to_lower_case(lst):

    lstx = list(map(to_lower_case, lst))

    return lstx    

    



def getZipcode_City(cell, noZip):

    if not noZip:

        cell = str(cell)

        regExp = re.findall("[\d]{3}-[\d]{4}", cell)

        try:

            zipCode = regExp[0] 

        except:

            zipCode = ''

        cellx = cell.replace(zipCode,'')

        city = cellx.split(",")[-1]

        city = city.strip()

        location = cell.replace(city,'')

        location = location.replace(zipCode,'')

        return [location, city, zipCode]

    else:

        cell = str(cell)

        city = cell.split(",")[-1]

        location = cell.replace(city,'')

        location = location.strip()

        return [location, city]



def Trandforme_date(stringDate) :

    months = {

    'January'   :1, 

    'February'  :2, 

    'March'     :3, 

    'April'     :4, 

    'May'       :5, 

    'June'      :6, 

    'July'      :7, 

    'August'    :8, 

    'September' :9, 

    'October'   :10, 

    'November'  :11, 

    'December'  :12,

    }



    if '-' in stringDate :

        return stringDate

    else :

        m = 0

        M_D = stringDate.split(',')[0]

        Y = stringDate.split(',')[1]

        M = M_D.split()[0]

        D = M_D.split()[1]



        for month, number in months.items() :

            if month.upper() == M.upper():

                m = number

        

        return Y+'-'+ str(m)+'-'+ D





# %%

#------------------------------------------------ Begin_Main ----------------------------------------

#data_list = []

for k, reg in enumerate(regdict):



    startDate=datetime.datetime.now()

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_")

    driver.get(regdict[reg])

    sleep(3)



    for time in range(10):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : Excel file = {os.listdir(tempfolder)})")

            file0 = os.path.join(tempfolder, os.listdir(tempfolder)[0])

            file_size = os.path.getsize(file0)



            if file_size == 0:

                os.remove(file0)   

                print(f'[ERROR] : Failed to Download Excel file {k+1}/{len(regdict)}. Try again To Download ' )

                sleep(2)

                driver.get(regdict[reg])

                continue

            else:

                break

        else:

            print(f"[INFO] : Download Excel file {k+1}/{len(regdict)}... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download Excel file {k+1}/{len(regdict)}. Run Script again' )





    Excel_file = os.listdir(tempfolder)[0]

    filePath = os.path.join(tempfolder, Excel_file)

    #data = pd.read_excel(key) #case where you have excel files with only one sheet each

    datax = pd.read_excel(filePath, index_col = None, header = None, sheet_name = None)#gives a collection of ordered dictionary

    NrOfSheets = len(datax)

    NrOfSheets_str = ''

    for i in range(NrOfSheets):

        #print(f"[INFO] : Processing sheet N° ", i)

        if NrOfSheets > 1: NrOfSheets_str = '_' + str(i+1)

        #datax = pd.read_excel(keys[0], index_col = 0, sheet_name = i)

        #data_list.append(datax)

        

        # go as deep as 7 steps to find the real headers of the table(or dataframe)

        title_index = 0

        lastUpdate = ''

        pd_seriex = datax[list(datax.keys())[i]]

        for row_i in range(min(7,len(pd_seriex)-1)):

            pd_serie_row = pd_seriex.iloc[row_i]

            for row_elt in pd_serie_row:

                if isStartingRowIndex(row_elt): title_index = row_i

                checkUpdate = getLastUpdateDate(row_elt)

                if checkUpdate != None: lastUpdate = checkUpdate

        #print(f"[INFO] : row number of the real headers: ", title_index)

        #print(f"[INFO] : Updated: ",lastUpdate)

        dfx = pd.DataFrame(datax[list(datax.keys())[i]], index = None)

        #print(dfx.head())

        #print("---------------------------------------------")

        new_serie = dfx.iloc[title_index:,:]

        j_series = list_to_lower_case(new_serie.iloc[0])

        

        #print(j_series)

        try:

            col_index = j_series.index("address")

            #print(f"[INFO] : index of address column: ", col_index)



            noZip = True

            for i in range(min(4,len(new_serie)-1)):

                cell = new_serie.iloc[i+1,col_index]

                #print(cell)

                #print("------")

                regExp = re.findall("[\d]{3}-[\d]{4}", str(cell))

                if len(regExp)>0:

                    #print(regExp)

                    noZip = False

                    break

            if noZip:

                # using apply function to create a new column

                '''

                City = [str(new_serie.iloc[row_index,col_index]).split(",")[-1] for row_index in range(len(new_serie))]

                new_serie[len(new_serie.columns)] = City

                #new_serie[len(new_serie.iloc[0])] = new_serie.apply(lambda row: str(row.index).split(",")[-1], axis = 1)

                #new_serie_2 = new_serie.assign(added = [str(new_serie.iloc[row_index][col_index]).split(",")[-1] for row_index in range(len(new_serie))])



                new_serie[len(new_serie.iloc[0])] = new_serie.apply(lambda row: lastUpdate, axis = 1)

                '''



                data = {"Location" : ["Address"], "City" : ["City"], "LastUpdated" : ["Update_Date"]}

                for i in range(len(new_serie)-1):

                    cell = str(new_serie.iloc[i+1,col_index])

                    address = getZipcode_City(str(cell), noZip)

                    for j in range(2): data[list(data.keys())[j]].append(address[j])

                    data["LastUpdated"].append(lastUpdate)

            else:



                data = {"Location" : ["Address"], "City" : ["City"], "ZipCode" : ["Zip"], "LastUpdated" : ["Update_Date"]}

                #create a dict to store the data from parse address cells

                for i in range(len(new_serie)-1):

                    cell = str(new_serie.iloc[i+1,col_index])

                    address = getZipcode_City(str(cell), noZip)

                    for j in range(3): data[list(data.keys())[j]].append(address[j])

                    data["LastUpdated"].append(lastUpdate)

                # append the data from the cell in the dictionary created

                # now create a dataframe from the dict and concatenate it with the previous one (new_serie)

                complement_serie = pd.DataFrame(data)

            for keyx in list(data.keys()):

                #print(data[key])

                new_serie[len(new_serie.columns)] = data[keyx]

            #print(new_serie.head())

            new_serie.drop(col_index, axis = 1, inplace = True)

        except:

            print(f"[ERROR] : No address information present on this sheet")

        #df = pd.DataFrame(data[list(data.keys())[i]])#

        #new_serie.to_excel(writer, sheet_name = Excel_file + NrOfSheets_str, header = False, index = False)

        # data_list.append(new_serie)

        # HeadersList = []

        # for i , data_ in enumerate( data_list):

        Headers = list(new_serie.iloc[0])

        #print(i,"| " ,Headers)

        for j, item in enumerate(Headers) :

            item = str(item)

            if ('name' in item) or ('Name' in item):

                if item != "name(Japanese)":

                    Headers[j] = 'NameInstitution'

            if ('phone' in item) or ('Phone' in item):

                Headers[j] = 'phone'

            

            if ('numbers' in item) :

                Headers[j] = 'RegistrationNumber'

            if item =='nan' :

                Headers[j] = 'Nan'



        new_serie.columns = Headers

        df = new_serie[1:]

        df = df.reset_index(drop=True)

        df = df.dropna(subset=["NameInstitution"])

        #data_list.append(df)



        for index, row in df.iterrows() :

      

            if (str(row['NameInstitution']) != 'nan') and (str(row['NameInstitution']) != '') and (str(row['NameInstitution']) != 'name'):

                

                sqldict['Name'].append(str(row['NameInstitution']))

                sqldict['Phone'].append(str(row['phone']))

                sqldict['ListProcessDate'].append(processdate)

                sqldict['RegCtry'].append(reg.split(' ')[0]) 

                sqldict['RegCode'].append(reg.split(' ')[1])

                sqldict['ListCode'].append(reg.split(' ')[-1]) 

                sqldict['Typology'].append(Typology[reg])

                sqldict['RegulationType'].append('Licensed')





                if 'Address' in Headers :

                    sqldict['Address_1'].append(str(row['Address']))

                else:

                    sqldict['Address_1'].append('')

                

                if 'City' in Headers :

                    sqldict['City'].append(str(row['City']))

                else:

                    sqldict['City'].append('')



                if 'Zip' in Headers :

                    sqldict['Zip'].append(str(row['Zip']))

                else:

                    sqldict['Zip'].append('')



                if 'Update_Date' in Headers :

                    sqldict['RegulationDate'].append( Trandforme_date(str(row['Update_Date']).strip()) )

                else:

                    sqldict['RegulationDate'].append('')

                    



                if 'JCN' in Headers :

                    if (str(row['JCN']) != 'nan') :

                        sqldict['InternalID_1'].append(str(row['JCN']))

                        sqldict['InternalID_1_type'].append('JCN')

                    else :

                        sqldict['InternalID_1'].append('')

                        sqldict['InternalID_1_type'].append('')

                else:

                    sqldict['InternalID_1'].append('')

                    sqldict['InternalID_1_type'].append('')



                if 'RegistrationNumber' in Headers :

                    sqldict['InternalID_2'].append(str(row['RegistrationNumber']))

                    sqldict['InternalID_2_type'].append('Registration numbers')

                else:

                    sqldict['InternalID_2'].append('')

                    sqldict['InternalID_2_type'].append('')

                    

        sqldict = bourange_same_length_array(sqldict)

        

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

        



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)


    
    
    